# GrepSeek Container Endpoint Demo

This notebook is a lightweight client for a GrepSeek vLLM server that is already running from the container image. It does not pull images, start Docker, start Apptainer, or launch `vllm serve`.

Start the server from the repository root on a GPU machine first:

```bash
MODEL_PATH=alireza7/GrepSeek-Qwen3.5-9B-GRPO PORT=10730 TP_SIZE=1 bash containers/serve_vllm.sh
```

Then point this notebook at the server endpoint.

In [ ]:
BASE_URL = "http://<host>:10730/v1"
MODEL = "grepseek"
API_KEY = "EMPTY"

print("BASE_URL:", BASE_URL)
print("MODEL:", MODEL)

## Check The Endpoint

In [ ]:
import json
import urllib.request

with urllib.request.urlopen(f"{BASE_URL}/models", timeout=15) as response:
    payload = json.loads(response.read().decode("utf-8"))

print(json.dumps(payload, indent=2)[:2000])
assert any(item.get("id") == MODEL for item in payload.get("data", [])), payload

## Send One Chat Request

In [ ]:
%pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Reply with exactly: The inference server is ready."}
    ],
    temperature=0,
    max_tokens=64,
)

print(response.choices[0].message.content)

## Optional: Run The Repository Sample Harness

Run this only in a checked-out GrepSeek repository. It uses the container wrapper to execute one sample question against the endpoint.

In [ ]:
import os
from pathlib import Path

if Path("containers/run_sample_inference.sh").exists():
    os.environ["BASE_URL"] = BASE_URL
    os.environ["MODEL"] = MODEL
    os.environ["API_KEY"] = API_KEY
    !bash containers/run_sample_inference.sh
else:
    print("Run this cell from the GrepSeek repository root to execute the sample harness.")